# Confirmação multiseed de DR-IF versus DR-IFT

Repetiremos os dois braços nas seeds 1337 e 2026. A seed 42 já foi concluída. Dentro de cada seed, dados e aleatoriedade são pareados; entre seeds, eles variam. Teste interno e FUSeg continuam bloqueados.


## 1. Preparar Colab e cache local


In [ ]:
from google.colab import drive

drive.mount('/content/drive')
from pathlib import Path
import os, shutil, subprocess, sys, yaml

PROJECT_ROOT = Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
DATA_ROOT = Path('/content/tcc_data')
%cd {PROJECT_ROOT}
!pip -q install albumentations pyyaml pytest

assert PROJECT_ROOT.is_dir(), f'Projeto ausente: {PROJECT_ROOT}'
source = PROJECT_ROOT / 'Dataset'
assert source.is_dir(), f'Dataset ausente: {source}'
shutil.copytree(source, DATA_ROOT / 'Dataset', dirs_exist_ok=True)
assert (DATA_ROOT / 'Dataset/DFUC2022_train_images/100001.jpg').is_file()
print('Cache preparado:', DATA_ROOT / 'Dataset')
print(
    'GPU:',
    subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True
    ).stdout.strip(),
)


In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"\nreturncode={code}")
    if code:
        raise subprocess.CalledProcessError(code, command)


def run_arm(config_path):
    run_stream(
        [
            sys.executable,
            "scripts/run_baseline_d0.py",
            "--project-root",
            str(PROJECT_ROOT),
            "--data-root",
            str(DATA_ROOT),
            "--config",
            config_path,
        ]
    )


## 2. Validar o contrato

Esta célula verifica arquitetura, seeds, streams de augmentation, fases progressivas e a igualdade dos controles dentro de cada par.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_fine_tuning.py",
        "tests/test_multiseed_confirmation.py",
        "tests/test_deep_res_unet_resnet34.py",
    ],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 3. Seed 1337 — DR-IF

Encoder fixo. Estimativa: aproximadamente 1,5–2 horas na mesma classe de GPU.


In [ ]:
run_arm("configs/fine_tuning_frozen_encoder_seed1337.yaml")


## 4. Seed 1337 — DR-IFT

Mesmo fluxo aleatório do braço anterior; somente a política de liberação muda.


In [ ]:
run_arm("configs/fine_tuning_progressive_seed1337.yaml")


## 5. Seed 2026 — DR-IF

Nova inicialização e nova sequência de augmentation, compartilhadas com seu par.


In [ ]:
run_arm("configs/fine_tuning_frozen_encoder_seed2026.yaml")


## 6. Seed 2026 — DR-IFT


In [ ]:
run_arm("configs/fine_tuning_progressive_seed2026.yaml")


## 7. Conferir e empacotar os quatro resultados

Após conferir a conclusão de cada execução, a célula gera quatro arquivos ZIP.


In [ ]:
import json

matrix = [
    ("seed1337", "DRIF"),
    ("seed1337", "DRIFT"),
    ("seed2026", "DRIF"),
    ("seed2026", "DRIFT"),
]
export_dir = PROJECT_ROOT / "exports/day11"
export_dir.mkdir(parents=True, exist_ok=True)
for seed_name, arm in matrix:
    root = PROJECT_ROOT / "results/day11" / seed_name / arm
    runs = sorted([p for p in root.glob("*") if p.is_dir()])
    assert runs, f"Nenhuma execução: {root}"
    run = runs[-1]
    summary = json.loads((run / "final_summary.json").read_text())
    assert summary["status"] == "completed"
    assert not summary["protected_partitions_accessed"]
    destination = export_dir / f"{arm}_{seed_name}"
    archive = shutil.make_archive(str(destination), "zip", run)
    print(archive)


## 8. Artefatos da execução

Os quatro arquivos ZIP ficam em `exports/day11`. Esta etapa usa a validação do DFUC2022. A síntese reúne as sementes 42, 1337 e 2026 sem tratar as 882 observações imagem-seed como independentes.
